# Trigger Components

> Trigger button and progress button for job monitor.

In [ ]:
#| default_exp components.trigger

In [ ]:
#| export
from typing import Optional

from fasthtml.common import Div, Button, Span, Script, FT

from cjm_fasthtml_daisyui.components.feedback.loading import loading, loading_styles, loading_sizes
from cjm_fasthtml_tailwind.utilities.spacing import m
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, items, gap
from cjm_fasthtml_tailwind.core.base import combine_classes

# Design system recipes (V1 button roles, V11 icon-size roles)
from cjm_fasthtml_design_system.buttons import buttons
from cjm_fasthtml_design_system.icons import icons

from cjm_fasthtml_job_monitor.html_ids import JobMonitorHtmlIds
from cjm_fasthtml_job_monitor.models import JobMonitorUrls, JobMonitorConfig

## render_job_trigger

The initial trigger button that submits a job when clicked.

In [ ]:
#| export
def render_job_trigger(
    config: JobMonitorConfig,     # Display config
    ids: JobMonitorHtmlIds,       # Element IDs
    urls: JobMonitorUrls,         # Route URLs
    disabled: bool = False,       # Disable button
    icon_fn: Optional[callable] = None,  # Icon renderer fn(name, **kwargs) -> FT
) -> FT:  # Trigger button wrapped in slot div
    """Render the initial trigger button."""
    btn_children = []
    if config.trigger_icon and icon_fn:
        btn_children.append(icon_fn(config.trigger_icon, size=icons.text_button))
    btn_children.append(Span(config.trigger_label))

    return Div(
        Button(
            *btn_children,
            hx_post=urls.trigger,
            hx_swap="none",
            disabled=disabled,
            cls=combine_classes(
                buttons.primary_action,
                flex_display, items.center, gap(1),
            ),
        ),
        id=ids.trigger_slot,
    )

In [ ]:
# Test render_job_trigger
from cjm_fasthtml_job_monitor.html_ids import JobMonitorHtmlIds
from cjm_fasthtml_job_monitor.models import JobMonitorUrls, JobMonitorConfig

ids = JobMonitorHtmlIds(prefix="jm")
urls = JobMonitorUrls(trigger="/fa/trigger", progress="/fa/progress", cancel="/fa/cancel")
config = JobMonitorConfig(trigger_label="Force Align")

result = render_job_trigger(config, ids, urls)
assert result.attrs['id'] == 'jm-trigger-slot'

# Find the button child
btn_el = result.children[0]
assert btn_el.tag == 'button'
assert btn_el.attrs.get('hx-post') == '/fa/trigger'
assert btn_el.attrs.get('hx-swap') == 'none'
# V1 buttons.primary_action role — solid primary, sm
assert 'btn' in btn_el.attrs['class']
assert 'btn-primary' in btn_el.attrs['class']
assert 'btn-sm' in btn_el.attrs['class']
print("render_job_trigger: OK")

In [ ]:
# Test render_job_trigger with icon
def mock_icon(name, **kwargs):
    return Span(f"[{name}]")

config_icon = JobMonitorConfig(trigger_label="Force Align", trigger_icon="audio-waveform")
result_icon = render_job_trigger(config_icon, ids, urls, icon_fn=mock_icon)
btn_el = result_icon.children[0]
# Should have icon span + label span
assert len(btn_el.children) == 2
print("render_job_trigger with icon: OK")

render_job_trigger with icon: OK


## render_job_progress_button

Shown in the trigger slot when the modal is closed during execution.
Clicking it reopens the modal.

In [ ]:
#| export
def render_job_progress_button(
    config: JobMonitorConfig,   # Display config
    ids: JobMonitorHtmlIds,     # Element IDs
) -> FT:  # Progress button wrapped in slot div
    """Render 'View Progress' button with spinner."""
    return Div(
        Button(
            Span(cls=combine_classes(loading, loading_styles.spinner, loading_sizes.xs)),
            Span(config.progress_label),
            type="button",  # Prevent form submission when inside StepFlow form wrapper
            onclick=f"document.getElementById('{ids.modal}').showModal()",
            cls=combine_classes(
                buttons.primary_action,
                flex_display, items.center, gap(1),
            ),
        ),
        id=ids.trigger_slot,
    )

In [ ]:
# Test render_job_progress_button
result = render_job_progress_button(config, ids)
assert result.attrs['id'] == 'jm-trigger-slot'
btn_el = result.children[0]
assert 'btn-primary' in btn_el.attrs['class']
assert 'showModal' in btn_el.attrs.get('onclick', '')
# Spinner + label
assert len(btn_el.children) == 2
spinner = btn_el.children[0]
assert 'loading' in spinner.attrs['class']
assert 'loading-spinner' in spinner.attrs['class']
print("render_job_progress_button: OK")

render_job_progress_button: OK


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()